In [1]:
import pandas as pd
import pyodbc
from sqlalchemy import create_engine,text
import urllib

<h2>Load Cleaned data</h2>

In [47]:
customers=pd.read_csv("..\\data\\cleaned\\olist_customers_dataset_cleaned.csv")
Geolocation=pd.read_csv("..\\data\\cleaned\\olist_geolocation_dataset_cleaned.csv")
CustomerOrderDetails=pd.read_csv("..\\data\\cleaned\\olist_orders_dataset_cleaned.csv")
ProductPaymentDetails=pd.read_csv("..\\data\\cleaned\\olist_order_payments_dataset_cleaned.csv")
OrderReview=pd.read_csv("..\\data\\cleaned\\olist_order_reviews_dataset_cleaned.csv")
ProductCategory=pd.read_csv("..\\data\\cleaned\\product_category_name_translation_cleaned.csv")
ProductDetails=pd.read_csv("..\\data\\cleaned\\olist_products_dataset_cleaned.csv")
SellerDetails=pd.read_csv("..\\data\\cleaned\\olist_sellers_dataset_cleaned.csv")
OrderItems=pd.read_csv("..\\data\\cleaned\\olist_order_items_dataset_cleaned.csv")

In [79]:
OrderItems.dtypes

order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

In [3]:
date_columns = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

CustomerOrderDetails[date_columns] = CustomerOrderDetails[date_columns].apply(
    pd.to_datetime
)

In [4]:
date_columns = [
    "review_creation_date",
    "review_answer_timestamp"
]

OrderReview[date_columns] = OrderReview[date_columns].apply(
    pd.to_datetime
)

In [5]:
OrderItems["shipping_limit_date"] = OrderItems["shipping_limit_date"].apply(pd.to_datetime)

<h2>Create database</h2>

In [64]:
connection_string = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=master;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

engine = create_engine(
    "mssql+pyodbc:///?odbc_connect=" + urllib.parse.quote_plus(connection_string)
)

with engine.connect() as conn:
    conn.execution_options(isolation_level="AUTOCOMMIT").execute(
        text("CREATE DATABASE Guviproject")
    )

print("Database created successfully!")

Database created successfully!


<h2>Connect the newly created database</h3>

In [6]:
connection_string = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=Guviproject;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

engine = create_engine(
    "mssql+pyodbc:///?odbc_connect=" + urllib.parse.quote_plus(connection_string)
)
print("Database connected")

Database connected


<h2>Create table using Cleaned data</h3>

In [13]:
customers

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409.0,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790.0,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151.0,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775.0,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056.0,campinas,SP
...,...,...,...,...,...
99436,17ddf5dd5d51696bb3d7c6291687be6f,1a29b476fee25c95fbafc67c5ac95cf8,3937.0,sao paulo,SP
99437,e7b71a9017aa05c9a7fd292d714858e8,d52a67c98be1cf6a5c84435bd38d095d,6764.0,taboao da serra,SP
99438,5e28dfe12db7fb50a4b2f691faecea5e,e9f50caf99f032f0bf3c55141f019d99,60115.0,fortaleza,CE
99439,56b18e2166679b8a959d72dd06da27f9,73c2643a0a458b49f58cea58833b192e,92120.0,canoas,RS


In [48]:
SellerDetails.to_sql(
    "olist_sellers_dataset",
    engine,
    if_exists="append",
    index=False
)

475

In [66]:
customers.to_sql(
    "olist_customers_dataset",
    engine,
    if_exists="append",
    index=False
)

Geolocation.to_sql(
    "olist_geolocation_dataset",
    engine,
    if_exists="append",
    index=False
)

CustomerOrderDetails.to_sql(
    "olist_orders_dataset",
    engine,
    if_exists="append",
    index=False
)

ProductPaymentDetails.to_sql(
    "olist_order_payments_dataset",
    engine,
    if_exists="append",
    index=False
)

OrderReview.to_sql(
    "olist_order_reviews_dataset",
    engine,
    if_exists="append",
    index=False
)
ProductCategory.to_sql(
    "product_category_name_translation",
    engine,
    if_exists="append",
    index=False
)
ProductDetails.to_sql(
    "olist_products_dataset",
    engine,
    if_exists="append",
    index=False
)
SellerDetails.to_sql(
    "olist_sellers_dataset",
    engine,
    if_exists="append",
    index=False
)
OrderItems.to_sql(
    "olist_order_items_dataset",
    engine,
    if_exists="append",
    index=False
)

226

<h2>Apply Primary key and foreign key constraints</h2>

<h3>Geolocation</h3>

In [9]:
Geolocation.dtypes

geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object

In [80]:
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE olist_geolocation_dataset
        ALTER COLUMN geolocation_zip_code_prefix int NOT NULL
    """))

    conn.execute(text("""
        ALTER TABLE olist_geolocation_dataset
        ADD CONSTRAINT PK_geolocation_zip_code_prefix
        PRIMARY KEY (geolocation_zip_code_prefix)
    """))

<h3>Customer

In [16]:
customers.dtypes

customer_id                     str
customer_unique_id              str
customer_zip_code_prefix    float64
customer_city                   str
customer_state                  str
dtype: object

In [ ]:
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE olist_customers_dataset
        ALTER COLUMN customer_id varchar(50) NOT NULL
    """))
    
    conn.execute(text("""
        ALTER TABLE olist_customers_dataset
        ADD CONSTRAINT PK_Customers
        PRIMARY KEY (customer_id)
    """))

In [15]:
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE olist_customers_dataset
        ADD CONSTRAINT FK_customer_zip_code_prefix
        FOREIGN KEY (customer_zip_code_prefix)
        REFERENCES olist_geolocation_dataset(geolocation_zip_code_prefix)
    """))

<h3>Order dataset</h3>

In [18]:
CustomerOrderDetails.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='str')

In [19]:
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE olist_orders_dataset
        ALTER COLUMN order_id varchar(50) NOT NULL
    """))
    
    conn.execute(text("""
        ALTER TABLE olist_orders_dataset
        ADD CONSTRAINT PK_order_id
        PRIMARY KEY (order_id)
    """))

In [21]:
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE olist_orders_dataset
        ALTER COLUMN customer_id varchar(50) 
    """))
    conn.execute(text("""
        ALTER TABLE olist_orders_dataset
        ADD CONSTRAINT FK_customer_id
        FOREIGN KEY (customer_id)
        REFERENCES olist_customers_dataset(customer_id)
    """))

<h3>Order Payment</h3>

In [23]:
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE olist_order_payments_dataset
        ALTER COLUMN order_id varchar(50) NOT NULL
    """))
    conn.execute(text("""
        ALTER TABLE olist_order_payments_dataset
        ALTER COLUMN payment_sequential int NOT NULL
    """))
    conn.execute(text("""
        ALTER TABLE olist_order_payments_dataset
        ADD CONSTRAINT CPK_order_id
        PRIMARY KEY (order_id,payment_sequential)
    """))

In [24]:
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE olist_order_payments_dataset
        ADD CONSTRAINT FK_order_id
        FOREIGN KEY (order_id)
        REFERENCES olist_orders_dataset(order_id)
    """))

<h3>Order Review</h3>

In [ ]:
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE olist_order_reviews_dataset
        ALTER COLUMN review_id varchar(50) NOT NULL
    """))
    conn.execute(text("""
        ALTER TABLE olist_order_reviews_dataset
        ALTER COLUMN order_id varchar(50) NOT NULL
    """))
    conn.execute(text("""
        ALTER TABLE olist_order_reviews_dataset
        ADD CONSTRAINT CPK_review_id
        PRIMARY KEY (review_id,order_id)
    """))

In [33]:
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE olist_order_reviews_dataset
        ADD CONSTRAINT FK_rev_order_id
        FOREIGN KEY (order_id)
        REFERENCES olist_orders_dataset(order_id)
    """))

<h3>Product Category</h3>

In [35]:
ProductCategory["product_category_name"].duplicated().sum()

np.int64(0)

In [38]:
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE product_category_name_translation
        ALTER COLUMN product_category_name varchar(100) NOT NULL
    """))
    conn.execute(text("""
        ALTER TABLE product_category_name_translation
        ADD CONSTRAINT PK_product_category_name
        PRIMARY KEY (product_category_name)
    """))


<h3>Product</h3>   

In [39]:
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE olist_products_dataset
        ALTER COLUMN product_id varchar(50) NOT NULL
    """))
    conn.execute(text("""
        ALTER TABLE olist_products_dataset
        ADD CONSTRAINT PK_product_id
        PRIMARY KEY (product_id)
    """))


In [44]:
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE olist_products_dataset
        ADD CONSTRAINT FK_product_category_name
        FOREIGN KEY (product_category_name)
        REFERENCES product_category_name_translation(product_category_name)
    """))

<h3>Seller</h3>

In [45]:
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE olist_sellers_dataset
        ALTER COLUMN seller_id varchar(50) NOT NULL
    """))
    conn.execute(text("""
        ALTER TABLE olist_sellers_dataset
        ADD CONSTRAINT PK_seller_id
        PRIMARY KEY (seller_id)
    """))


In [49]:
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE olist_sellers_dataset
        ADD CONSTRAINT FK_seller_loc
        FOREIGN KEY (seller_zip_code_prefix)
        REFERENCES olist_geolocation_dataset(geolocation_zip_code_prefix)
    """))

<h3>Order Items</h3>

In [53]:

with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE olist_order_items_dataset
        ALTER COLUMN order_item_id int NOT NULL
    """))
    conn.execute(text("""
        ALTER TABLE olist_order_items_dataset
        ALTER COLUMN order_id varchar(50) NOT NULL
    """))
    conn.execute(text("""
        ALTER TABLE olist_order_items_dataset
        ADD CONSTRAINT PK_order_item_id
        PRIMARY KEY (order_item_id,order_id)
    """))

In [54]:
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE olist_order_items_dataset
        ADD CONSTRAINT FK_order_item_id
        FOREIGN KEY (order_id)
        REFERENCES olist_orders_dataset(order_id)
    """))

In [55]:
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE olist_order_items_dataset
        ALTER COLUMN product_id varchar(50)
    """))
    conn.execute(text("""
        ALTER TABLE olist_order_items_dataset
        ADD CONSTRAINT FK_product_id
        FOREIGN KEY (product_id)
        REFERENCES olist_products_dataset(product_id)
    """))

In [56]:
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE olist_order_items_dataset
        ALTER COLUMN seller_id varchar(50)
    """))
    conn.execute(text("""
        ALTER TABLE olist_order_items_dataset
        ADD CONSTRAINT FK_seller_id
        FOREIGN KEY (seller_id)
        REFERENCES olist_sellers_dataset(seller_id)
    """))